#### Getting started With Langchain And Open AI

In this quickstart we'll see how to:

- Get setup with LangChain, LangSmith and LangServe
- Use the most basic and common components of LangChain: prompt templates, models, and output parsers.
- Build a simple application with LangChain
- Trace your application with LangSmith
- Serve your application with LangServe

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENROUTER_API_KEY']=os.getenv("OPENROUTER_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="openai/gpt-oss-120b:free")
print(llm)

profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001E5DE9105C0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E583BB7380> root_client=<openai.OpenAI object at 0x000001E583AD52E0> root_async_client=<openai.AsyncOpenAI object at 0x000001E583B38B00> model_name='openai/gpt-oss-120b:free' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [4]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
     model="openai/gpt-oss-20b:free",
    messages=[{"role": "user", "content": "What is Generative AI?"}]
)

print(response.choices[0].message.content)

**Generative AI** refers to a class of artificial‑intelligence models that learn to *create* new data that resembles the data they were trained on. Unlike traditional “discriminative” models that simply classify or predict labels (e.g., “is this image a cat?”), generative models learn the underlying distribution of the data and can sample from that distribution to produce brand‑new examples—text, images, audio, code, 3‑D shapes, and more.

---

## Core Concepts

| Concept | What it means | Example |
|---------|---------------|---------|
| **Training data** | A large collection of real examples (e.g., books, photos, songs). | 8 million web pages for GPT‑4 |
| **Learning the distribution** | The model adjusts internal parameters so that the probability of the training data is maximized. | Neural network weights that capture grammar patterns |
| **Sampling / generation** | Drawing new samples from the learned distribution. | Writing a new paragraph that sounds like a Shakespeare sonnet |


In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [8]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="openai/gpt-oss-20b:free")
print(llm)

profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001E583D22A20> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E583D238C0> root_client=<openai.OpenAI object at 0x000001E583CF7050> root_async_client=<openai.AsyncOpenAI object at 0x000001E583D228A0> model_name='openai/gpt-oss-20b:free' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [11]:
## Input and get response form LLM

result=llm.invoke("What is generative AI?")
#https://smith.langchain.com/o/367def88-4b63-4d4c-bb10-a5ab8ab1bae2

In [12]:
print(result)

content='**Generative AI** is a branch of artificial intelligence that learns to *create* new data that resembles the data it was trained on.  \nInstead of simply classifying or predicting an outcome (the job of “discriminative” models), generative models learn the underlying distribution of a dataset and can sample from that distribution to produce novel examples.\n\n---\n\n## Core Concepts\n\n| Concept | What it means | Typical example |\n|---------|---------------|-----------------|\n| **Training objective** | Learn a probability distribution \\(P(x)\\) over data \\(x\\). | Maximize likelihood of training samples. |\n| **Generation** | Sample from the learned distribution to produce new data. | Generate a new image, sentence, or music clip. |\n| **Latent space** | A compressed, often lower‑dimensional representation of data. | A 512‑dimensional vector that can be decoded into an image. |\n| **Discriminative vs. Generative** | Discriminative models predict labels \\(P(y|x)\\); genera

In [13]:
### Chatprompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an expert AI Engineer. Provide me answers based on the questions"),
        ("user","{input}")
    ]

)
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Provide me answers based on the questions'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [14]:
## chain 
chain=prompt|llm

response=chain.invoke({"input":"Can you tell me about Langsmith?"})
print(response)

content='**Langsmith** is a lightweight, open‑source platform built by the LangChain team to help developers build, monitor, debug, and evaluate large‑language‑model (LLM) applications. Think of it as a “telemetry hub” that sits between your code and the LLM provider, collecting rich runtime data and making it easy to inspect, analyze, and improve your workflows.\n\n---\n\n## 1. Core Purpose\n\n| Goal | How Langsmith Helps |\n|------|---------------------|\n| **Observability** | Stores every request/response, token usage, latency, cost, and metadata in a searchable trace store. |\n| **Debugging** | Step‑by‑step view of a chain’s execution, with the ability to replay or modify inputs. |\n| **Evaluation** | Run automated tests against your LLM logic, compare outputs, and compute metrics (BLEU, ROUGE, custom scoring). |\n| **Monitoring** | Aggregate metrics (latency, cost, token count) and set alerts for SLA violations or budget limits. |\n| **Collaboration** | Share traces and dashboards

In [15]:
type(response)

langchain_core.messages.ai.AIMessage

In [16]:
## stroutput Parser

from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
chain=prompt|llm|output_parser

response=chain.invoke({"input":"Can you tell me about Langsmith?"})
print(response)

**Langsmith** is a purpose‑built observability platform for building, testing, and monitoring large‑language‑model (LLM) applications. It sits on top of the LangChain ecosystem but can be used with any LLM‑driven codebase.

---

## 1. What Langsmith Does

| Feature | What it gives you | Typical use case |
|---------|-------------------|-----------------|
| **Tracing** | Every call to an LLM, prompt, chain, or tool is recorded as a node in a trace. | Visualize the flow of a conversation or a multi‑step chain. |
| **Metrics** | Latency, token usage, cost, and custom metrics are automatically collected. | Spot performance bottlenecks or cost spikes. |
| **Logging** | Structured logs for every step, including inputs, outputs, and metadata. | Debug unexpected outputs or failures. |
| **Evaluation** | Run unit‑style tests against prompts or chains and compare results. | Ensure a prompt still works after a model update. |
| **Dashboards** | Web UI with trace explorer, metric charts, and alert